# T31 / E14 — Bậc thang kích thước mô hình đọc

**Chạy trong MỘT phiên.** Bản đầu của notebook này bắt chạy hai phiên riêng; không cần thiết, và
đây là lý do.

## Save Version có bị hạ xung không — có, nhưng nó chỉ đụng vào một cột

Hạ xung là chuyện của **phần cứng**, không phải của cách khởi chạy. Save Version chạy trên đúng
card T4 ấy qua papermill, nên nó **không tránh được** hạ xung. Mục 5 `CLAUDE.md` đo được T4 chậm
đi 10–15 % sau vài phút chạy liên tục.

Nhưng phải hỏi tiếp: hạ xung làm hỏng **cái gì**?

| Cột | Hạ xung có đụng tới không |
|---|---|
| macro-F1, F1 từng lớp, ECE | **không** — trọng số chú ý y hệt nhau dù card chạy 1.590 hay 1.200 MHz |
| ms mỗi mẫu | **có** — và đây là một nửa câu hỏi của E14 |

Nên gộp hai cỡ vào một phiên là an toàn cho toàn bộ phần độ chính xác. Chỉ cột chi phí cần xử lý
riêng, và mục 5 `CLAUDE.md` cho sẵn cách: *"chạy mỗi cấu hình một phiên riêng **hoặc đo xen kẽ**"*.

## Đo xen kẽ, tức ô 11

Ô 9 đo chi phí theo thứ tự **7B → 3B → 1.5B → 7B → 3B → 1.5B**, mỗi lượt một tiến trình riêng nạp lại mô
hình từ đầu. Nếu card có trôi trong phiên thì nó trôi lên **cả hai** mô hình như nhau thay vì dồn
hết vào mô hình chạy sau. Hai lượt của cùng một mô hình lệch nhau bao nhiêu chính là thước đo
mức trôi — in ra để đọc, không giấu.

`measure_throughput.py` còn đọc **nhiệt độ GPU** ở mỗi lượt, nên nếu hạ xung xảy ra thì thấy được
trực tiếp chứ không phải suy đoán.

## Chỉ phải chạy HAI nấc, không phải ba

| Nấc | Trích đặc trưng? | Đo chi phí? | Ghi chú |
|---|---|---|---|
| Qwen2.5-7B | **không** | **có** | Độ chính xác lấy từ E02 (0,7451) và E03 (0,7567) |
| Qwen2.5-3B | có | có | |
| Qwen2.5-1.5B | có | có | |

**Vì sao 7B không trích lại nhưng vẫn đo chi phí.** Hai nửa của E14 trả lời câu này khác nhau.
Độ chính xác của 7B đã có, cùng bộ dữ liệu và cùng cách chia đoạn, shard còn nằm sẵn trên máy —
trích lại là đốt 62 phút GPU để dựng lại con số đã có. Nhưng chi phí 528 ms/mẫu của nó đo ở một
**phiên khác** với phiên sắp đo 3B và 1.5B, tức đúng kiểu so sánh mà mục 5 `CLAUDE.md` bảo đừng
làm, và đúng hạn chế mà cột chi phí của E13 đã phải mang. Mười phút đo xen kẽ sửa được chuyện
đó.

## Hai câu E14 trả lời

1. **Câu cũ:** phương pháp chạy được trên phần cứng nhỏ hơn không, mất bao nhiêu điểm — trục chi
   phí của CH2.
2. **Câu mới, do T30 sinh ra:** E13 đo được nhóm chunk-aware **không chuyển** giữa Qwen2.5-7B và
   Sailor2-8B — hai mô hình khác họ huấn luyện. Bậc thang này hỏi nó có chuyển giữa các **cỡ của
   cùng một họ** không. Phép thử nhẹ hơn hẳn: cùng kiến trúc, cùng dữ liệu huấn luyện, chỉ khác
   số tham số.

Vì câu thứ hai, mỗi cỡ có **hai** cấu hình — một chunk-aware và một mốc lookback gộp của chính
nó. T30 dạy bài này bằng một kết luận sai: so chunk-aware của Sailor2 với mốc của Qwen thì chênh
lệch trộn hai biến, không tách được.

## Chi phí ước tính

| Bước | Thời gian |
|---|---|
| Dò kiểu số, hai cỡ | ~10 phút |
| Trích 3B, `float16` (7.000 mẫu) | ~60 phút |
| Trích 1.5B, `bfloat16` (7.000 mẫu) | ~130 phút |
| Đo chi phí xen kẽ, sáu lượt | ~30 phút |
| **Tổng** | **~4 giờ** cộng thời gian tải mô hình |

**Nấc 1.5B chạy `bfloat16`, khác hai nấc kia.** Ở `float16` nó tràn số trên cả 28 lớp, 20/20 mẫu — không còn lớp nào để bỏ. `bfloat16` cùng dải mũ với `float32` nên không tràn, nhưng T4 là Turing, không có bf16 gốc, nên chậm 4,1 lần. Đó là lý do nấc nhỏ nhất lại tốn nhiều giờ GPU nhất — và tự nó là một kết quả của E14: **nấc lùi này đắt hơn gấp đôi nấc 3B mà nó lùi khỏi** (1.101 so với 513 ms/mẫu).

**Chấm điểm chạy ở máy cá nhân**, theo quy tắc chốt ở T23.


In [1]:
# Ô 1 — lấy code. Chạy lại được nhiều lần.
import os
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/wsunicorn/vihallulens.git"
REPO_DIR = Path("/kaggle/working/vihallulens")


def run(*args, cwd=None):
    print("$", " ".join(str(a) for a in args))
    result = subprocess.run(args, cwd=cwd, capture_output=True, text=True)
    print(result.stdout.strip())
    if result.returncode:
        print(result.stderr.strip())
        raise SystemExit(f"lệnh hỏng: {' '.join(str(a) for a in args)}")
    return result.stdout


if REPO_DIR.exists():
    run("git", "fetch", "--all", cwd=REPO_DIR)
    run("git", "reset", "--hard", "origin/main", cwd=REPO_DIR)
else:
    run("git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR))

os.chdir(REPO_DIR)
run("git", "log", "-1", "--format=%h %s")

$ git clone --depth 1 https://github.com/wsunicorn/vihallulens.git /kaggle/working/vihallulens

$ git log -1 --format=%h %s
231fe3c T31: cất kết quả trích trước khi đo chi phí, và chặn thời gian mỗi lượt đo (#93)


'231fe3c T31: cất kết quả trích trước khi đo chi phí, và chặn thời gian mỗi lượt đo (#93)\n'

In [2]:
# Ô 2 — ba nấc của bậc thang. Không phải sửa gì ở ô này.
#
# Nac 7B co "trich": False. Ly do: E02 va E03 da trich no roi, cung bo du lieu, cung cach chia
# doan, cung nhom dac trung — trich lai la dot 62 phut GPU de dung lai con so da co.
#
# Nhung no VAN nam trong o 9. Chi phi 528 ms/mau cua 7B do o mot PHIEN KHAC voi phien sap do 3B
# va 1.5B, tuc dung kieu so sanh ma muc 5 CLAUDE.md bao dung lam. Muoi phut do xen ke sua duoc
# chuyen do va cho E11 mot duong cong chi phi that su la duong cong.
BAC_THANG = [
    {
        "co": "7B",
        "mo_hinh": "Qwen/Qwen2.5-7B-Instruct",
        "chunk": "configs/e03_chunk_sentence_vihallu.yaml",
        "moc": None,
        "trich": False,
    },
    {
        "co": "3B",
        "mo_hinh": "Qwen/Qwen2.5-3B-Instruct",
        "chunk": "configs/e14_qwen3b_vihallu.yaml",
        "moc": "configs/e14_baseline_lookback_qwen3b.yaml",
        "trich": True,
    },
    {
        "co": "1.5B",
        "mo_hinh": "Qwen/Qwen2.5-1.5B-Instruct",
        "chunk": "configs/e14_qwen15b_vihallu.yaml",
        "moc": "configs/e14_baseline_lookback_qwen15b.yaml",
        "trich": True,
    },
]
CAN_TRICH = [n for n in BAC_THANG if n["trich"]]

print("=" * 78)
for nac in BAC_THANG:
    viec = "trich + do chi phi" if nac["trich"] else "CHI do chi phi (da co dac trung)"
    print(f"  {nac['co']:<6} {nac['mo_hinh']:<30} {viec}")
print("=" * 78)
print(f"  Trich {len(CAN_TRICH)} nac, do chi phi ca {len(BAC_THANG)} nac.")

  7B     Qwen/Qwen2.5-7B-Instruct       CHI do chi phi (da co dac trung)
  3B     Qwen/Qwen2.5-3B-Instruct       trich + do chi phi
  1.5B   Qwen/Qwen2.5-1.5B-Instruct     trich + do chi phi
  Trich 2 nac, do chi phi ca 3 nac.


In [3]:
# Ô 3 — cài đặt. bitsandbytes cần cho lượng tử hóa 4 bit.
!pip install -q --no-deps -e .
!pip install -q bitsandbytes

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for vihallulens (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 36.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
vihallulens 0.1.0 requires pyvi, which is not installed.
vihallulens 0.1.0 requires rank-bm25, which is not installed.


In [4]:
# Ô 4 — TIỀN KIỂM. Vài giây, chạy trước mọi thứ.
import importlib.util
import sys
from pathlib import Path

sys.path.insert(0, "src")
from vihallulens.config import load_config
from vihallulens.data.paths import find_raw_dir

problems = []

packages = ("torch", "transformers", "bitsandbytes", "pandas", "accelerate")
absent = [name for name in packages if importlib.util.find_spec(name) is None]
trang_thai = f"THIEU {absent}" if absent else f"du ca {list(packages)}"
print(f"  goi phai co san   : {trang_thai}")
if absent:
    problems.append(f"thieu goi {absent}")

try:
    raw = find_raw_dir()
    files = sorted(p.name for p in Path(raw).glob("vihallu*"))
    print(f"  du lieu tho       : {raw}")
    print(f"  file vihallu tho  : {files or 'KHONG CO'}")
    if not files:
        problems.append("khong thay file vihallu nao trong du lieu tho")
except Exception as error:
    print(f"  du lieu tho       : KHONG TIM THAY ({error})")
    problems.append("chua mount dataset du lieu tho")

for nac in BAC_THANG:
    khoas = ("chunk", "moc") if nac["trich"] else ("chunk",)
    for khoa in khoas:
        cfg = load_config(nac[khoa])
        ghi_chu = "<- o 6 se ghi de" if nac["trich"] else "<- da chot tu T07, khong dung toi"
        print(f"  {nac['co']:<6} {khoa:<6}: {Path(nac[khoa]).name:<38} "
              f"exclude_layers {str(cfg.extractor.exclude_layers):<8} {ghi_chu}")

chain = [
    ("o 5", "data/interim/vihallu_{train,dev,test}.parquet", "normalize_data + split_data"),
    ("o 6", "exclude_layers trong BON cau hinh cua 3B va 1.5B", "compare_dtypes, hai luot"),
    ("o 7", "hai hash cua moi co phai trung nhau", "kiem"),
    ("o 8", "data/processed/vihallu_{split}_<hash>.jsonl", "extract_features"),
    ("o 10", "ket_qua_t31/ — CAT KET QUA DAT TIEN DI TRUOC", "shutil.copy"),
    ("o 11", "chi phi ms/mau ca ba nac, do xen ke", "measure_throughput, sau luot"),
]
print("  chuoi tu tao:")
for cell, target, maker in chain:
    print(f"    {cell:<7} {target:<46} <- {maker}")

if problems:
    raise SystemExit("TIEN KIEM HONG: " + "; ".join(problems))
print("\nTien kiem dat.")

  goi phai co san   : du ca ['torch', 'transformers', 'bitsandbytes', 'pandas', 'accelerate']
  du lieu tho       : /kaggle/input/datasets/unicorn1209/vihallulens
  file vihallu tho  : ['vihallu_test_public.csv', 'vihallu_train.csv']
  7B     chunk : e03_chunk_sentence_vihallu.yaml        exclude_layers [27]     <- da chot tu T07, khong dung toi
  3B     chunk : e14_qwen3b_vihallu.yaml                exclude_layers []       <- o 6 se ghi de
  3B     moc   : e14_baseline_lookback_qwen3b.yaml      exclude_layers []       <- o 6 se ghi de
  1.5B   chunk : e14_qwen15b_vihallu.yaml               exclude_layers []       <- o 6 se ghi de
  1.5B   moc   : e14_baseline_lookback_qwen15b.yaml     exclude_layers []       <- o 6 se ghi de
  chuoi tu tao:
    o 5     data/interim/vihallu_{train,dev,test}.parquet  <- normalize_data + split_data
    o 6     exclude_layers trong BON cau hinh cua 3B va 1.5B <- compare_dtypes, hai luot
    o 7     hai hash cua moi co phai trung nhau            <- kiem
  

In [5]:
# Ô 5 — chuẩn bị dữ liệu và kiểm môi trường. Khoảng 2 phút, CPU.
!python scripts/probe_env.py
!python scripts/normalize_data.py --dataset vihallu
!python scripts/split_data.py --only vihallu
!python -m pytest tests/test_attention_hook.py tests/test_drop_nonfinite.py -q


MÔI TRƯỜNG
  repo             : /kaggle/working/vihallulens
  commit           : 231fe3c T31: cất kết quả trích trước khi đo chi phí, và chặn thời gian mỗi lượt đo (#93)
  python           : 3.12.13
  torch            : 2.10.0+cu128
  transformers     : 5.0.0
  bitsandbytes     : 0.50.2
  accelerate       : 1.13.0
  vihallulens      : 0.1.0 tại /kaggle/working/vihallulens/src/vihallulens/__init__.py
  dữ liệu          : /kaggle/input/datasets/unicorn1209/vihallulens  (14 file)
      MANIFEST.md
      isedsc01_test_private.json
      isedsc01_test_public.json
      isedsc01_train.json
      vifactcheck_dataset_card.md
      vifactcheck_dev.parquet
      vifactcheck_gitattributes.txt
      vifactcheck_test.parquet
      vifactcheck_train.parquet
      vihallu_test_public.csv
      vihallu_train.csv
      viwikifc_dev.csv
      viwikifc_test.csv
      viwikifc_train.csv

CHUẨN HÓA VIHALLU
  nguồn                 : /kaggle/input/datasets/unicorn1209/vihallulens
  số dòng               : 7

## Dò lớp tràn số — vẫn phải đo cho từng cỡ

Qwen2.5-7B hỏng đúng lớp 27, nhưng hai cỡ này là mô hình khác và T30 cho thấy chuyện này **không
suy ra được**: Sailor2 hỏng theo kiểu hoàn toàn khác — cả mạng hỏng trên 0,7 % mẫu thay vì một
lớp hỏng trên mọi mẫu.

Ô 6 ghi kết quả vào **cả bốn** cấu hình. Ô 7 kiểm hai hash của mỗi cỡ có trùng nhau không — lệch
là mốc lookback sẽ đòi trích lại từ đầu thay vì dùng lại shard.

In [6]:
# Ô 6 — DÒ LỚP TRÀN SỐ, hai cỡ. Khoảng 10 phút GPU. BẮT BUỘC chạy trước ô 8.
#
# Qwen2.5-7B hong dung lop 27, nhung hai co nay la mo hinh khac, so lop khac. T30 cho thay chuyen
# nay KHONG suy ra duoc: Sailor2 hong theo kieu hoan toan khac Qwen.
#
# Hai mo hinh nay nho hon Sailor2 nhieu nen luot moc bfloat16 se vua bo nho — lan nay co ca phan
# kiem "cac lop con song co bi bop meo khong" ma Sailor2 khong cho duoc.
import ast
import os
import re
import sys
from pathlib import Path

sys.path.insert(0, "src")
from vihallulens.config import load_config

os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
LOP_TRAN = {}
NAC_HONG = {}

for nac in list(CAN_TRICH):
    print("=" * 78)
    print(f"  DO KIEU SO — {nac['co']}  ({nac['mo_hinh']})")
    print("=" * 78)
    # Kieu so lay tu YAML chu khong quyet dinh o day — config la nguon su that, notebook chi
    # doc lai. Nac 1.5B de compute_dtype: bfloat16 vi float16 tran so ca 28 lop.
    kieu = load_config(nac["chunk"]).extractor.compute_dtype
    # Moc so: float32 sach nhat nhung nap ton bo nho nhat. Voi nac chay bfloat16 thi mo hinh du
    # nho de float32 vua, nen nac do co con so |delta| do theo float32 that — do hon cai can tren
    # ma nac 3B phai chiu khi lay bfloat16 lam moc.
    moc_kieu = "bfloat16" if kieu == "float16" else "float32"
    print(f"  kieu so dang xet: {kieu}   moc so: {moc_kieu}")
    proc = subprocess.run(
        ["python", "scripts/compare_dtypes.py", "--model", nac["mo_hinh"],
         "--per-dataset", "10", "--dtype", kieu, "--reference", moc_kieu],
        capture_output=True, text=True, env={**os.environ},
    )
    print(proc.stdout[-7000:])

    # Mot nac hong KHONG duoc keo ca phien theo. Luot 09/09: nac 3B chay xong sach se, roi nac
    # 1.5B hong, va dong `raise` o day nuot luon bon o con lai cua 3B — gan mot gio GPU doi lay
    # mot cau tra loi roi vut di phan bien no thanh dac trung. Nac hong thi ghi lai va bo ra
    # khoi CAN_TRICH; cac nac khac di tiep.
    ly_do = None
    if proc.returncode:
        print(proc.stderr[-3000:])
        ly_do = f"compare_dtypes tra ve ma loi {proc.returncode}"

    found = re.search(r"^EXCLUDE_LAYERS=(\[.*\])$", proc.stdout, flags=re.MULTILINE)
    tong = re.search(r"^N_LAYERS=(\d+)$", proc.stdout, flags=re.MULTILINE)
    if ly_do is None and not (found and tong):
        ly_do = "khong thay dong EXCLUDE_LAYERS= hoac N_LAYERS= trong output"

    bad = sorted(set(ast.literal_eval(found.group(1)))) if found else []
    if ly_do is None and len(bad) >= int(tong.group(1)):
        # Bo het lop thi khong con gi de trich — AttentionExtractor bao loi ngay luc khoi tao.
        # Day khong phai loi cong cu, day la ket qua: mo hinh nay khong doc duoc o float16.
        ly_do = f"tran so o TAT CA {tong.group(1)} lop — khong doc duoc o {kieu}"

    if ly_do:
        NAC_HONG[nac["co"]] = ly_do
        CAN_TRICH = [n for n in CAN_TRICH if n["co"] != nac["co"]]
        print(f"\n  !! {nac['co']}: {ly_do}")
        print("     Bo nac nay ra khoi phan trich. Cac nac con lai van chay tiep binh thuong.")
        continue

    LOP_TRAN[nac["co"]] = bad
    print(f"\n  {nac['co']} ({kieu}): lop tran so = {bad if bad else 'KHONG CO'}")
    if not bad:
        print("    Voi Qwen2.5-7B thi lop cuoi luon tran, nen ket qua nay dang chu y — no nghia")
        print("    la mo hinh nho hon KHONG thua huong loi tran so cua 7B. Doc bang per-layer o")
        print("    tren de xac nhan, roi giu nguyen exclude_layers rong.")

    # Ghi vao CA HAI cau hinh cua co nay. Thieu mot cai thi hai hash lech nhau va moc lookback
    # se doi trich lai tu dau thay vi dung lai shard.
    for khoa in ("chunk", "moc"):
        path = Path(nac[khoa])
        text = path.read_text(encoding="utf-8")
        # Kiem DONG CO TON TAI, khong kiem "text co doi khong". Hai chuyen khac nhau:
        #   dong khong co trong file   -> hong that, phai dung
        #   thay xong ma khong doi gi  -> bad rong, ket qua hop le, phai chay tiep
        # Luot 09/09 chet o day vi Qwen2.5-3B khong co lop nao tran, bad = [], nen phep thay ghi
        # "exclude_layers: []" de len "exclude_layers: []" — y het nhau.
        if not re.search(r"^  exclude_layers: \[\]$", text, flags=re.MULTILINE):
            raise SystemExit(
                f"khong tim thay dong 'exclude_layers: []' trong {nac[khoa]}. "
                f"Co the o nay da chay roi trong phien nay, hoac file config bi sua tay."
            )
        patched = re.sub(r"^  exclude_layers: \[\]$", f"  exclude_layers: {bad}", text, count=1,
                         flags=re.MULTILINE)
        path.write_text(patched, encoding="utf-8")
        trang_thai = "giu nguyen []" if not bad else f"-> {bad}"
        print(f"    da ghi: {nac[khoa]:<44} {trang_thai}")

print()
print(f"  Tom tat lop tran so: {LOP_TRAN}")
if NAC_HONG:
    print()
    print("  " + "!" * 74)
    for co, ly_do in NAC_HONG.items():
        print(f"  NAC HONG — {co}: {ly_do}")
    con_lai = ", ".join(n["co"] for n in CAN_TRICH) or "khong nac nao"
    print(f"  O 7, 8, 10, 11 tu day chi chay cho: {con_lai}")
    print("  O 9 VAN do chi phi du ca ba nac: ms/mau khong phu thuoc vao viec dac trung co huu")
    print("  han hay khong, cung ngan ay phep nhan ma tran. Nhung con so float16 cua mot nac")
    print("  hong KHONG phai gia phai tra that, vi muon dung nac do thi phai doi kieu so.")
    print("  " + "!" * 74)
if not CAN_TRICH:
    print()
    print("  Khong con nac nao de trich. Bo qua o 8, 10, 11; van chay o 9 de lay cot chi phi.")
print("  NHO commit lai cac file config sau khi chay xong.")

  DO KIEU SO — 3B  (Qwen/Qwen2.5-3B-Instruct)
  kieu so dang xet: float16   moc so: bfloat16

SO SÁNH FLOAT16 VỚI BFLOAT16
  dữ liệu   : /kaggle/input/datasets/unicorn1209/vihallulens
  số mẫu    : 20
  độ dài    : 47 đến 4805 từ
    [float16] 1/20
    [float16] 2/20
    [float16] 3/20
    [float16] 4/20
    [float16] 5/20
    [float16] 6/20
    [float16] 7/20
    [float16] 8/20
    [float16] 9/20
    [float16] 10/20
    [float16] 11/20
    [float16] 12/20
    [float16] 13/20
    [float16] 14/20
    [float16] 15/20
    [float16] 16/20
    [float16] 17/20
    [float16] 18/20
    [float16] 19/20
    [float16] 20/20
    [float16] xong 20 mẫu trong 11 s
    [bfloat16] 1/20
    [bfloat16] 2/20
    [bfloat16] 3/20
    [bfloat16] 4/20
    [bfloat16] 5/20
    [bfloat16] 6/20
    [bfloat16] 7/20
    [bfloat16] 8/20
    [bfloat16] 9/20
    [bfloat16] 10/20
    [bfloat16] 11/20
    [bfloat16] 12/20
    [bfloat16] 13/20
    [bfloat16] 14/20
    [bfloat16] 15/20
    [bfloat16] 16/20
    [bfloat16] 

In [7]:
# Ô 7 — cổng kiểm trước khi tiêu GPU. Vài giây, CPU.
import sys

sys.path.insert(0, "src")
from importlib import reload

import vihallulens.config as config_module

reload(config_module)
HASH = {}
for nac in CAN_TRICH:
    chunk = config_module.load_config(nac["chunk"])
    moc = config_module.load_config(nac["moc"])
    h_chunk = config_module.extraction_hash(chunk)
    h_moc = config_module.extraction_hash(moc)
    HASH[nac["co"]] = h_chunk
    print(f"  {nac['co']:<6} exclude {str(chunk.extractor.exclude_layers):<12} "
          f"hash chunk {h_chunk}  hash moc {h_moc}")
    if not chunk.extractor.exclude_layers and not moc.extractor.exclude_layers:
        print("    (ca hai deu trong — chi dung neu o 6 do duoc that su khong lop nao tran)")
    if h_chunk != h_moc:
        raise SystemExit(
            f"NAC {nac['co']}: HAI HASH KHAC NHAU. Moc lookback se doi trich lai tu dau thay vi "
            f"dung lai shard. Nguyen nhan gan nhu chac chan la o 6 chi ghi duoc vao mot file."
        )

if len(set(HASH.values())) != len(HASH):
    raise SystemExit(f"HAI NAC RA CUNG MOT HASH: {HASH}. Kiem lai model_name trong config.")
if not HASH:
    print("\n  Khong con nac nao qua duoc o 6, nen khong co hash nao de kiem.")
else:
    print(f"\n  {len(HASH)} nac, hash khac nhau, moi nac hai cau hinh trung hash: {HASH}")

  3B     exclude []           hash chunk 8f77142aa220  hash moc 8f77142aa220
    (ca hai deu trong — chi dung neu o 6 do duoc that su khong lop nao tran)
  1.5B   exclude []           hash chunk 9e2b80984a73  hash moc 9e2b80984a73
    (ca hai deu trong — chi dung neu o 6 do duoc that su khong lop nao tran)

  2 nac, hash khac nhau, moi nac hai cau hinh trung hash: {'3B': '8f77142aa220', '1.5B': '9e2b80984a73'}


## Trích đặc trưng

**Đọc gì trong lúc chạy:** dòng `lỗi` phải là 0, và dòng `LỚP TRÀN SỐ` nếu xuất hiện thì đọc kỹ —
nó liệt kê lớp nào tràn và bao nhiêu mẫu. Liệt kê *mọi* lớp thì đó là kiểu hỏng của Sailor2, xử
lý bằng bỏ mẫu chứ không bỏ lớp.

In [8]:
# Ô 8 — trích đặc trưng hai cỡ nhỏ. Khoảng 1 giờ 40. Chạy lại được, có lưu tiến độ.
# Nac 1.5B chiem gan het cho do: bfloat16 cham hon float16 tren T4 (Turing khong co bf16 goc).
import os

os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["TRANSFORMERS_VERBOSITY"] = "error"
# Khong dem output: mot o chay hang tieng dong ho ma log chi hien sau 13 phut thi khong phan biet
# duoc "dang chay" voi "treo". Luot 09/09 mat ca dem vi khong nhin thay diem dung.
os.environ["PYTHONUNBUFFERED"] = "1"

if not CAN_TRICH:
    print("Khong con nac nao qua duoc o 6 — khong trich gi ca. Chay tiep o 9.")

HONG_TRICH = []
for nac in CAN_TRICH:
    print("=" * 78)
    print(f"  TRICH — {nac['co']}")
    print("=" * 78)
    for split in ("train", "dev", "test"):
        code = os.system(
            f"python scripts/extract_features.py --config {nac['chunk']} --split {split}")
        if code:
            # KHONG raise. Trich co luu tien do, nen mot shard do dang van dang mang ve — phien
            # sau doc tiep tu cho do thay vi chay lai tu dau. Raise o day thi o 9 va o 10 khong
            # chay, va cai do dang bi bo lai tren may Kaggle. Dung bai hoc T30.
            HONG_TRICH.append(f"{nac['co']}/{split} (ma loi {code})")
            print(f"  !! {nac['co']}/{split} hong, ma loi {code} — ghi lai va di tiep")

if HONG_TRICH:
    print()
    print("  " + "!" * 74)
    print(f"  CO {len(HONG_TRICH)} LUOT TRICH HONG: {', '.join(HONG_TRICH)}")
    print("  Van chay tiep o 9 va o 10 de mang ve phan da trich duoc.")
    print("  " + "!" * 74)

  TRICH — 3B

T20 — TRÍCH ĐẶC TRƯNG LOOKBACK
  cấu hình              : configs/e14_qwen3b_vihallu.yaml  (hash 8f77142aa220)
  mô hình đọc           : Qwen/Qwen2.5-3B-Instruct
  lượng tử hóa / kiểu số: nf4 / float16
  bỏ lớp                : không bỏ lớp nào
  trần token ngữ cảnh   : 4,096
  chia đoạn             : sentence
  bộ dữ liệu            : vihallu/train, 5,600 mẫu
  khối đặc trưng ghi ra : lookback_total, lookback_context, chunk_entropy, chunk_max_share, chunk_gini, top1_top2_gap, chunk_drift
  mẫu có bằng chứng     : 0/5,600
  đã có sẵn             : 0 mẫu trong vihallu_train_8f77142aa220.jsonl
  còn phải chạy         : 5,600 mẫu


       5600/5600  218 ms/mẫu  còn ~0 phút  lỗi 0

--------------------------------------------------------------------------------
  đã ghi thêm           : 5,600 mẫu, tổng 5,600
  lỗi                   : 0
  bị cắt ngữ cảnh       : 0/5,600
  có lớp tràn số        : 0/5,600
  thời gian             : 20.3 phút, 218 ms/mẫu
  file                  : data/processed/vihallu_train_8f77142aa220.jsonl

T20 — TRÍCH ĐẶC TRƯNG LOOKBACK
  cấu hình              : configs/e14_qwen3b_vihallu.yaml  (hash 8f77142aa220)
  mô hình đọc           : Qwen/Qwen2.5-3B-Instruct
  lượng tử hóa / kiểu số: nf4 / float16
  bỏ lớp                : không bỏ lớp nào
  trần token ngữ cảnh   : 4,096
  chia đoạn             : sentence
  bộ dữ liệu            : vihallu/dev, 700 mẫu
  khối đặc trưng ghi ra : lookback_total, lookback_context, chunk_entropy, chunk_max_share, chunk_gini, top1_top2_gap, chunk_drift
  mẫu có bằng chứng     : 0/700
  đã có sẵn             : 0 mẫu trong vihallu_dev_8f77142aa220.jsonl
  còn phải c

        700/700  218 ms/mẫu  còn ~0 phút  lỗi 0

--------------------------------------------------------------------------------
  đã ghi thêm           : 700 mẫu, tổng 700
  lỗi                   : 0
  bị cắt ngữ cảnh       : 0/700
  có lớp tràn số        : 0/700
  thời gian             : 2.5 phút, 218 ms/mẫu
  file                  : data/processed/vihallu_dev_8f77142aa220.jsonl

T20 — TRÍCH ĐẶC TRƯNG LOOKBACK
  cấu hình              : configs/e14_qwen3b_vihallu.yaml  (hash 8f77142aa220)
  mô hình đọc           : Qwen/Qwen2.5-3B-Instruct
  lượng tử hóa / kiểu số: nf4 / float16
  bỏ lớp                : không bỏ lớp nào
  trần token ngữ cảnh   : 4,096
  chia đoạn             : sentence
  bộ dữ liệu            : vihallu/test, 700 mẫu
  khối đặc trưng ghi ra : lookback_total, lookback_context, chunk_entropy, chunk_max_share, chunk_gini, top1_top2_gap, chunk_drift
  mẫu có bằng chứng     : 0/700
  đã có sẵn             : 0 mẫu trong vihallu_test_8f77142aa220.jsonl
  còn phải chạy       

        700/700  221 ms/mẫu  còn ~0 phút  lỗi 0

--------------------------------------------------------------------------------
  đã ghi thêm           : 700 mẫu, tổng 700
  lỗi                   : 0
  bị cắt ngữ cảnh       : 0/700
  có lớp tràn số        : 0/700
  thời gian             : 2.6 phút, 221 ms/mẫu
  file                  : data/processed/vihallu_test_8f77142aa220.jsonl
  TRICH — 1.5B

T20 — TRÍCH ĐẶC TRƯNG LOOKBACK
  cấu hình              : configs/e14_qwen15b_vihallu.yaml  (hash 9e2b80984a73)
  mô hình đọc           : Qwen/Qwen2.5-1.5B-Instruct
  lượng tử hóa / kiểu số: nf4 / bfloat16
  bỏ lớp                : không bỏ lớp nào
  trần token ngữ cảnh   : 4,096
  chia đoạn             : sentence
  bộ dữ liệu            : vihallu/train, 5,600 mẫu
  khối đặc trưng ghi ra : lookback_total, lookback_context, chunk_entropy, chunk_max_share, chunk_gini, top1_top2_gap, chunk_drift
  mẫu có bằng chứng     : 0/5,600
  đã có sẵn             : 0 mẫu trong vihallu_train_9e2b80984a73.js

       5600/5600  551 ms/mẫu  còn ~0 phút  lỗi 0

--------------------------------------------------------------------------------
  đã ghi thêm           : 5,600 mẫu, tổng 5,600
  lỗi                   : 0
  bị cắt ngữ cảnh       : 0/5,600
  có lớp tràn số        : 0/5,600
  thời gian             : 51.4 phút, 551 ms/mẫu
  file                  : data/processed/vihallu_train_9e2b80984a73.jsonl

T20 — TRÍCH ĐẶC TRƯNG LOOKBACK
  cấu hình              : configs/e14_qwen15b_vihallu.yaml  (hash 9e2b80984a73)
  mô hình đọc           : Qwen/Qwen2.5-1.5B-Instruct
  lượng tử hóa / kiểu số: nf4 / bfloat16
  bỏ lớp                : không bỏ lớp nào
  trần token ngữ cảnh   : 4,096
  chia đoạn             : sentence
  bộ dữ liệu            : vihallu/dev, 700 mẫu
  khối đặc trưng ghi ra : lookback_total, lookback_context, chunk_entropy, chunk_max_share, chunk_gini, top1_top2_gap, chunk_drift
  mẫu có bằng chứng     : 0/700
  đã có sẵn             : 0 mẫu trong vihallu_dev_9e2b80984a73.jsonl
  còn ph

        700/700  549 ms/mẫu  còn ~0 phút  lỗi 0

--------------------------------------------------------------------------------
  đã ghi thêm           : 700 mẫu, tổng 700
  lỗi                   : 0
  bị cắt ngữ cảnh       : 0/700
  có lớp tràn số        : 0/700
  thời gian             : 6.4 phút, 549 ms/mẫu
  file                  : data/processed/vihallu_dev_9e2b80984a73.jsonl

T20 — TRÍCH ĐẶC TRƯNG LOOKBACK
  cấu hình              : configs/e14_qwen15b_vihallu.yaml  (hash 9e2b80984a73)
  mô hình đọc           : Qwen/Qwen2.5-1.5B-Instruct
  lượng tử hóa / kiểu số: nf4 / bfloat16
  bỏ lớp                : không bỏ lớp nào
  trần token ngữ cảnh   : 4,096
  chia đoạn             : sentence
  bộ dữ liệu            : vihallu/test, 700 mẫu
  khối đặc trưng ghi ra : lookback_total, lookback_context, chunk_entropy, chunk_max_share, chunk_gini, top1_top2_gap, chunk_drift
  mẫu có bằng chứng     : 0/700
  đã có sẵn             : 0 mẫu trong vihallu_test_9e2b80984a73.jsonl
  còn phải chạy   

        700/700  555 ms/mẫu  còn ~0 phút  lỗi 0

--------------------------------------------------------------------------------
  đã ghi thêm           : 700 mẫu, tổng 700
  lỗi                   : 0
  bị cắt ngữ cảnh       : 0/700
  có lớp tràn số        : 0/700
  thời gian             : 6.5 phút, 555 ms/mẫu
  file                  : data/processed/vihallu_test_9e2b80984a73.jsonl


In [9]:
# Ô 9 — soi shard hai cỡ vừa trích. Vài giây, CPU. KHÔNG dừng notebook.
#
# Khong raise du shard co nan: mot shard hong la thu CAN dem ve nhat de chan doan. Bai hoc T30.
for nac in CAN_TRICH:
    print("=" * 78)
    print(f"  SOI SHARD — {nac['co']}")
    print("=" * 78)
    proc = subprocess.run(["python", "scripts/inspect_shard.py", "--config", nac["chunk"]],
                          capture_output=True, text=True)
    print(proc.stdout[-6000:])
    if proc.returncode:
        print(proc.stderr[-2000:])

print()
print("  DU SHARD CO NAN HAY KHONG, VAN CHAY O 11 DE MANG VE.")

  SOI SHARD — 3B

SOI SHARD — e14_qwen3b_vihallu
  mô hình đọc       : Qwen/Qwen2.5-3B-Instruct
  đang bỏ lớp       : []
  hash trích        : 8f77142aa220

  TRAIN  5,600 mẫu   lưới 36 × 16
    mẫu có lớp tràn số : 0  (0.00 %)

  DEV  700 mẫu   lưới 36 × 16
    mẫu có lớp tràn số : 0  (0.00 %)

  TEST  700 mẫu   lưới 36 × 16
    mẫu có lớp tràn số : 0  (0.00 %)

------------------------------------------------------------------------------------
  Không lớp nào tràn số. Shard sạch.

  SOI SHARD — 1.5B

SOI SHARD — e14_qwen15b_vihallu
  mô hình đọc       : Qwen/Qwen2.5-1.5B-Instruct
  đang bỏ lớp       : []
  hash trích        : 9e2b80984a73

  TRAIN  5,600 mẫu   lưới 28 × 12
    mẫu có lớp tràn số : 0  (0.00 %)

  DEV  700 mẫu   lưới 28 × 12
    mẫu có lớp tràn số : 0  (0.00 %)

  TEST  700 mẫu   lưới 28 × 12
    mẫu có lớp tràn số : 0  (0.00 %)

------------------------------------------------------------------------------------
  Không lớp nào tràn số. Shard sạch.


  DU SHARD CO NA

## Chấm điểm — KHÔNG chạy ở đây

Quy tắc chốt ở T23: mọi phép so sánh phải chấm trên **cùng một máy**, vì điểm dev lệch tới 0,0075
giữa Kaggle và máy cá nhân do bộ giải tối ưu hội tụ khác nhau.

In [10]:
# Ô 10 — LẤY KẾT QUẢ VỀ. Vài giây. Ô này chạy TRƯỚC ô đo chi phí, cố ý.
#
# Trich xong la thu dat nhat trong ca phien va khong mua lai duoc bang gi ngoai GPU. Do chi phi
# thi re va chay lai luc nao cung duoc. Nen cat cai dat di truoc, roi moi lam cai re.
#
# Luot 09/09 lam nguoc lai: o do chi phi treo o nac 1.5B, va 99 phut trich khong bao gio ra toi
# ket_qua_t31. T30 va T26 deu dat o "lay ket qua ve" ngay sau o trich, khong chen viec GPU nao o
# giua — day la quay lai dung nep do.
#
# O 11 chay xong se goi lai o nay mot lan nua de cap nhat runs.jsonl.
import shutil
import sys
from pathlib import Path

sys.path.insert(0, "src")
from vihallulens.config import extraction_hash, load_config

out = Path("/kaggle/working/ket_qua_t31")
out.mkdir(exist_ok=True)
for nac in CAN_TRICH:
    run = extraction_hash(load_config(nac["chunk"]))
    for split in ("train", "dev", "test"):
        src = Path(f"data/processed/vihallu_{split}_{run}.jsonl")
        if src.exists():
            shutil.copy(src, out / src.name)
    for khoa in ("chunk", "moc"):
        shutil.copy(nac[khoa], out / Path(nac[khoa]).name)
if Path("results/runs.jsonl").exists():
    shutil.copy("results/runs.jsonl", out / "runs_t31.jsonl")

for f in sorted(out.iterdir()):
    print(f"  {f.name:<44} {f.stat().st_size / 1e6:>8.1f} MB")

print("""
Tai het thu muc ket_qua_t31 ve may, dat vao:
  *.jsonl (vihallu_*)  ->  data/processed/
  *.yaml               ->  configs/   (GHI DE — chung mang exclude_layers da do)
  runs_t31.jsonl       ->  giu lai, no chua sau luot do chi phi cua o 11

Roi bao lai de cham diem o may ca nhan, 0 giay GPU.
""")

  e14_baseline_lookback_qwen15b.yaml                0.0 MB
  e14_baseline_lookback_qwen3b.yaml                 0.0 MB
  e14_qwen15b_vihallu.yaml                          0.0 MB
  e14_qwen3b_vihallu.yaml                           0.0 MB
  runs_t31.jsonl                                    0.4 MB
  vihallu_dev_8f77142aa220.jsonl                   28.2 MB
  vihallu_dev_9e2b80984a73.jsonl                   16.6 MB
  vihallu_test_8f77142aa220.jsonl                  28.2 MB
  vihallu_test_9e2b80984a73.jsonl                  16.6 MB
  vihallu_train_8f77142aa220.jsonl                225.0 MB
  vihallu_train_9e2b80984a73.jsonl                132.4 MB

Tai het thu muc ket_qua_t31 ve may, dat vao:
  *.jsonl (vihallu_*)  ->  data/processed/
  *.yaml               ->  configs/   (GHI DE — chung mang exclude_layers da do)
  runs_t31.jsonl       ->  giu lai, no chua sau luot do chi phi cua o 11

Roi bao lai de cham diem o may ca nhan, 0 giay GPU.



## Đo chi phí xen kẽ

Ô này là thứ giữ cho cột `ms/mẫu` của E14 **so được**, dù hai cỡ chạy chung một phiên.

**Ô này để cuối cùng, cố ý.** Trích đặc trưng là thứ đắt nhất trong phiên và chỉ mua lại được
bằng giờ GPU; đo chi phí thì rẻ và chạy lại lúc nào cũng được. Nên ô 10 cất phần đắt đi trước,
rồi mới tới ô này. Lượt 09/09 làm ngược lại và một chỗ treo ở nấc 1.5B đã chôn theo 99 phút trích
— cùng nếp mà T30 và T26 đã đặt sẵn: không chèn việc GPU nào giữa ô trích và ô lấy kết quả về.

Mỗi lượt có hạn mức 15 phút. Quá hạn thì bỏ lượt đó và chạy lượt sau, không dừng cả ô.

Đọc kết quả bằng cách so **lần 1 với lần 2 của cùng một mô hình**. Hai lần lệch nhau bao nhiêu
chính là mức trôi của card trong phiên. Lệch nhỏ thì cột chi phí dùng được; lệch lớn thì vẫn dùng
được nhưng phải báo cáo mức trôi bên cạnh.

In [11]:
# Ô 11 — ĐO CHI PHÍ XEN KẼ, ca ba nac. Khoảng 30 phút GPU. Giu cho cot ms/mau so duoc.
#
# Thu tu: 7B -> 3B -> 1.5B -> 7B -> 3B -> 1.5B.
# Moi luot mot tien trinh rieng, nap lai mo hinh tu dau.
#
# Ly do khong do noi nhau tung mo hinh: T4 ha xung 10-15 % sau vai phut chay lien tuc (muc 5
# CLAUDE.md). Do noi nhau thi mo hinh chay sau luon co ve cham hon, va phan cham do bi tinh nham
# thanh khac biet giua hai mo hinh. Xen ke thi phan troi do roi len CA HAI nhu nhau.
#
# Hai luot cua cung mot mo hinh lech nhau bao nhieu CHINH LA thuoc do muc troi — doc no, dung bo
# qua. measure_throughput.py con doc nhiet do GPU moi luot.
import os
import shutil
import subprocess
import sys
from pathlib import Path

os.environ["PYTHONUNBUFFERED"] = "1"
sys.path.insert(0, "src")
from vihallulens.config import load_config

# Hai vong qua ca ba nac: 7B, 3B, 1.5B, roi lap lai. Card troi thi troi len ca ba nhu nhau.
THU_TU = BAC_THANG + BAC_THANG
# Moi luot mot han muc rieng. Luot 7B that su ton 269 giay, nen 15 phut la gap 3,3 lan —
# rong rai that, ma van chan duoc chuyen mot luot ket giu ca phien qua dem.
HAN_MUC_GIAY = 900
HONG_CHI_PHI = []

for vong, nac in enumerate(THU_TU, start=1):
    lan = 1 if vong <= len(BAC_THANG) else 2
    cfg = load_config(nac["chunk"])
    # Truyen co NAY DU DANH SACH RONG. measure_throughput.py de --exclude-layers mac dinh la
    # [27], nen khong truyen thi 3B va 1.5B lang le bi bo lop 27 — do chi phi tren 35/36 va
    # 27/28 lop trong khi phan trich dung du lop. Cot chi phi va cot do chinh xac se do tren hai
    # mang khac nhau ma khong ai thay. argparse voi nargs="*" nhan co rong, tra ve [].
    bo_lop = [str(x) for x in cfg.extractor.exclude_layers]
    # Moi nac do o DUNG kieu so no se chay that. Do 1.5B o float16 se ra 271 ms/mau cho mot cau
    # hinh khong sinh noi mot con so huu han — nhanh, va vo nghia.
    kieu = cfg.extractor.compute_dtype
    ten = f"t31_chiphi_{nac['co'].replace('.', '_')}_lan{lan}"
    print("=" * 78)
    print(f"  LUOT {vong}/{len(THU_TU)} — {nac['co']} [{kieu}], lan {lan}   run_name {ten}")
    print(f"  bo lop {bo_lop or 'khong bo lop nao'}   han muc {HAN_MUC_GIAY // 60} phut")
    print("=" * 78)
    lenh = ["python", "scripts/measure_throughput.py", "--model", nac["mo_hinh"],
            "--per-tier", "8", "--run-name", ten, "--compute-dtype", kieu,
            "--exclude-layers", *bo_lop]
    try:
        proc = subprocess.run(lenh, timeout=HAN_MUC_GIAY, env={**os.environ})
        if proc.returncode:
            HONG_CHI_PHI.append(f"luot {vong} ({nac['co']}, lan {lan}): ma loi {proc.returncode}")
            print(f"  !! luot {vong} tra ve ma loi {proc.returncode} — ghi lai va di tiep")
    except subprocess.TimeoutExpired:
        # Qua han thi bo luot do, dung dung ca o. Cac luot khac van cho so dung, va cot chi phi
        # thieu mot o con hon ca phien khong co gi.
        HONG_CHI_PHI.append(f"luot {vong} ({nac['co']}, lan {lan}): qua {HAN_MUC_GIAY // 60} phut")
        print(f"  !! luot {vong} qua han {HAN_MUC_GIAY // 60} phut — bo luot nay, chay luot sau")

if HONG_CHI_PHI:
    print()
    print("  " + "!" * 74)
    for dong in HONG_CHI_PHI:
        print(f"  {dong}")
    print("  Cot chi phi thieu nhung so trich thi khong sao — o 10 da mang no ve roi.")
    print("  " + "!" * 74)

# Cap nhat lai runs.jsonl trong ket_qua_t31: o 10 da chay truoc khi co cac dong chi phi nay.
_out = Path("/kaggle/working/ket_qua_t31")
if _out.is_dir() and Path("results/runs.jsonl").exists():
    shutil.copy("results/runs.jsonl", _out / "runs_t31.jsonl")
    print(f"\n  Da cap nhat {_out / 'runs_t31.jsonl'} voi cac dong chi phi vua do.")

print()
print("  Doc gi: so sanh lan 1 voi lan 2 CUA CUNG MOT MO HINH.")
print("  Lech nho  -> card khong troi dang ke, cot ms/mau so duoc.")
print("  Lech lon  -> card co troi, phai bao cao muc troi ben canh con so chi phi.")

  LUOT 1/6 — 7B [float16], lan 1   run_name t31_chiphi_7B_lan1
  bo lop ['27']   han muc 15 phut

T08 — ĐO THÔNG LƯỢNG VÀ QUYẾT ĐỊNH BẬC THANG
  mô hình               : Qwen/Qwen2.5-7B-Instruct
  ngân sách token       : 4096
  compute dtype         : float16
  lớp bỏ qua            : [27]
  dữ liệu               : /kaggle/input/datasets/unicorn1209/vihallulens


  nạp mô hình           : 152 s, 27 lớp được hook
  khung mẫu prompt      : 37 token, 41 token khi có câu hỏi

--------------------------------------------------------------------------------
PHÂN BỐ ĐỘ DÀI THEO MỨC — đếm trên toàn bộ mẫu có nhãn
--------------------------------------------------------------------------------
  Bộ                     0–512      513–1024     1025–2048     2049–4096      tổng
  --------------------------------------------------------------------------------
  vihallu                6,461           530             6             3     7,000
  isedsc01               6,188        17,770        11,931           480    36,369
  viwikifc              20,102           817             0             0    20,919
  vifactcheck              705         3,546         2,785           196     7,232
  --------------------------------------------------------------------------------
  tổng                  33,456        22,663        14,722           679    71,520

-------

  nạp mô hình           : 21 s, 36 lớp được hook
  khung mẫu prompt      : 37 token, 41 token khi có câu hỏi

--------------------------------------------------------------------------------
PHÂN BỐ ĐỘ DÀI THEO MỨC — đếm trên toàn bộ mẫu có nhãn
--------------------------------------------------------------------------------
  Bộ                     0–512      513–1024     1025–2048     2049–4096      tổng
  --------------------------------------------------------------------------------
  vihallu                6,461           530             6             3     7,000
  isedsc01               6,188        17,770        11,931           480    36,369
  viwikifc              20,102           817             0             0    20,919
  vifactcheck              705         3,546         2,785           196     7,232
  --------------------------------------------------------------------------------
  tổng                  33,456        22,663        14,722           679    71,520

--------

  nạp mô hình           : 10 s, 28 lớp được hook
  khung mẫu prompt      : 37 token, 41 token khi có câu hỏi

--------------------------------------------------------------------------------
PHÂN BỐ ĐỘ DÀI THEO MỨC — đếm trên toàn bộ mẫu có nhãn
--------------------------------------------------------------------------------
  Bộ                     0–512      513–1024     1025–2048     2049–4096      tổng
  --------------------------------------------------------------------------------
  vihallu                6,461           530             6             3     7,000
  isedsc01               6,188        17,770        11,931           480    36,369
  viwikifc              20,102           817             0             0    20,919
  vifactcheck              705         3,546         2,785           196     7,232
  --------------------------------------------------------------------------------
  tổng                  33,456        22,663        14,722           679    71,520

--------

  nạp mô hình           : 60 s, 27 lớp được hook
  khung mẫu prompt      : 37 token, 41 token khi có câu hỏi

--------------------------------------------------------------------------------
PHÂN BỐ ĐỘ DÀI THEO MỨC — đếm trên toàn bộ mẫu có nhãn
--------------------------------------------------------------------------------
  Bộ                     0–512      513–1024     1025–2048     2049–4096      tổng
  --------------------------------------------------------------------------------
  vihallu                6,461           530             6             3     7,000
  isedsc01               6,188        17,770        11,931           480    36,369
  viwikifc              20,102           817             0             0    20,919
  vifactcheck              705         3,546         2,785           196     7,232
  --------------------------------------------------------------------------------
  tổng                  33,456        22,663        14,722           679    71,520

--------

  nạp mô hình           : 43 s, 36 lớp được hook
  khung mẫu prompt      : 37 token, 41 token khi có câu hỏi

--------------------------------------------------------------------------------
PHÂN BỐ ĐỘ DÀI THEO MỨC — đếm trên toàn bộ mẫu có nhãn
--------------------------------------------------------------------------------
  Bộ                     0–512      513–1024     1025–2048     2049–4096      tổng
  --------------------------------------------------------------------------------
  vihallu                6,461           530             6             3     7,000
  isedsc01               6,188        17,770        11,931           480    36,369
  viwikifc              20,102           817             0             0    20,919
  vifactcheck              705         3,546         2,785           196     7,232
  --------------------------------------------------------------------------------
  tổng                  33,456        22,663        14,722           679    71,520

--------


  Da cap nhat /kaggle/working/ket_qua_t31/runs_t31.jsonl voi cac dong chi phi vua do.

  Doc gi: so sanh lan 1 voi lan 2 CUA CUNG MOT MO HINH.
  Lech nho  -> card khong troi dang ke, cot ms/mau so duoc.
  Lech lon  -> card co troi, phai bao cao muc troi ben canh con so chi phi.
